#### Ayriklastirma

Discretizasyon veya b?lme, s?rekli de?i?kenleri, de?i?ken de?erlerinin aral???n? kapsayan bir dizi biti?ik aral?k veya b?lme olu?turarak kesikli de?i?kenlere d?n??t?rme i?lemidir. Discretizasyon, carpik de?i?kenlerin(skewed) da??l?m?n? de?i?tirmek ve **ayk?r? de?erlerin etkisini en aza indirmek** ve bu ?ekilde baz? makine ??renimi modellerinin performans?n? art?rmak i?in kullan?l?r.

Discretizasyon, ayk?r? de?erlerin etkisini nas?l en aza indirir? Discretizasyon, ayk?r? de?erleri da??l?m?n alt veya ?st aral?klar?na, da??l?m?n geri kalan uygun de?erleriyle birlikte yerle?tirir. Bu nedenle, bu ayk?r? g?zlemler art?k da??l?m?n u?lar?ndaki di?er de?erlerden farkl?l?k g?stermez, ??nk? ?imdi hepsi ayn? aral?k veya b?lmede bir araya gelmi?tir. Ayr?ca, e?it s?kl?kl? b?l?mler aras?nda g?zlemleri s?ralarsan?z, discretizasyon, ?arp?k bir de?i?kenin de?erlerini de?er aral??? boyunca daha homojen bir ?ekilde yayarak da??t?r.

<img src='../images/discretization.png'>

[Kaynakca](https://blog.revolutionanalytics.com/2013/04/extending-revoscaler-for-mining-big-data-discretization.html)

In [ ]:
# To get multi output in jupyternotebook
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# sklearn ciktisini nnumpy array olarak donmektedir. Bu adimi pandas icin ayarlayabiliriz.
from sklearn import set_config
set_config(transform_output="pandas")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.model_selection import train_test_split

In [ ]:
data=pd.read_csv('../data/feature-engine/boston.csv')
data.describe()

In [ ]:
x=data.drop('MEDV', axis=1)
y=data[['MEDV']]
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.3, random_state=0)

#### Esit araliklarla

Surekli degisken icerisinde yer alan en buyuk ve en kucuk deger arasindaki farkin belirlenen bolme sayisina bolunerek bulanan araliklar esit olur. Degisken degerleri bu deger araliginda ise o aralik icine girer.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
discretizer= KBinsDiscretizer(n_bins=10, encode='ordinal',strategy='uniform')

discretizer.fit_transform(X_train[['LSTAT','NOX','DIS']])
discretizer.transform(X_test[['LSTAT','NOX','DIS']])

In [ ]:
tmp = discretizer.fit_transform(X_train[['LSTAT','NOX','DIS']])
tmp['LSTAT'].value_counts()

In [ ]:
discretizer.bin_edges_

#### Esit agirlikli

Yani, surekli degiskeni belirlenen bolme sayisina gore her bolmede esit sayida gozlem olacak sekilde ayarlamaya calisalim.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
discretizer= KBinsDiscretizer(n_bins=10, encode='ordinal',strategy='quantile')

discretizer.fit_transform(X_train[['LSTAT','NOX','DIS']])
discretizer.transform(X_test[['LSTAT','NOX','DIS']])

In [ ]:
tmp = discretizer.fit_transform(X_train[['LSTAT','NOX','DIS']])
tmp['LSTAT'].value_counts()

#### K-ortalamalar ile ayriklastirma

K-ortalamalar k?melemesi kullan?larak yap?lan ayr?kla?t?rmada, aral?klar k-ortalamalar algoritmas? taraf?ndan tan?mlanan k?melerdir. K?me say?s? (k) kullan?c? taraf?ndan tan?mlan?r. k-ortalamalar k?meleme algoritmas?n?n iki ana ad?m? vard?r. 

- Ba?langic ad?m?nda, k tane k?menin ba?lang?? merkezleri olarak rastgele se?ilir ve veri noktalar? en yak?n k?meye atan?r. 
- Iterasyon ad?m?nda, k?melerin merkezleri k?me i?indeki t?m g?zlemlerin ortalama noktalar? olarak yeniden hesaplan?r ve g?zlemler yeni olu?turulan en yak?n k?meye yeniden atan?r.

?terasyon ad?m? en uygun k merkez bulunana kadar devam eder. Interaktif bir gorsellestirmeyi [suradan](https://www.naftaliharris.com/blog/visualizing-k-means-clustering/) inceleyebilirsiniz.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
discretizer= KBinsDiscretizer(n_bins=10, encode='ordinal',strategy='kmeans')

discretizer.fit_transform(X_train[['LSTAT','NOX','DIS']])
discretizer.transform(X_test[['LSTAT','NOX','DIS']])

In [ ]:
tmp = discretizer.fit_transform(X_train[['LSTAT','NOX','DIS']])
tmp['LSTAT'].value_counts()

#### Karar Agaclari ile Ayriklastirma

Karar a?a?lar? ile ayr?kla?t?rma, de?i?ken de?erlerinin s?ralanaca?? en uygun b?lmeleri belirlemek i?in bir karar a?ac?n?n kullan?lmas?ndan olu?ur. Karar a?ac?, ayr?kla?t?r?lacak de?i?ken ve hedef kullan?larak olu?turulur. Bir karar a?ac? bir tahminde bulundu?unda, bir g?zlemi N u? yapraktan birine atar, bu nedenle herhangi bir karar a?ac?, de?erleri N yapra??n?n her birindeki tahminler olan ayr?k bir ??kt? ?retecektir. Karar a?a?lar? ile ayr?kla?t?rma, kutular ve hedef aras?nda monotonik bir ili?ki yarat?r.

Bu teknik, [KDD 2009](https://www.kdd.org/kdd-cup/view/kdd-cup-2009) veri bilimi yarismasinda *Winning the KDD Cup Orange Challenge with Ensemble Selection* adi altinda kullanailmistir.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.3, random_state=0)

from sklearn.tree import DecisionTreeRegressor


discretizer= DecisionTreeRegressor(max_depth=5, random_state=0)

discretizer.fit(X_train[['LSTAT']], y_train)
X_train['LSTAT_tree']=discretizer.predict(X_train[['LSTAT']])
X_test['LSTAT_tree']=discretizer.predict(X_test[['LSTAT']])

In [ ]:
X_train['LSTAT_tree'].unique()

In [ ]:
X_train['LSTAT_tree']